In [ ]:
# Install required libraries
!pip install -q --upgrade openai pinecone python-dotenv tiktoken


## Demo: RAG Pipeline with OpenRouter + Pinecone
In this short tutorial, you'll:
- Set up OpenRouter and Pinecone
- Index a tiny 3-document corpus
- Compare vanilla LLM vs RAG-augmented answers

OpenRouter provides an OpenAI-compatible API, so the same OpenAI Python SDK can be used with an OpenRouter API key.


### 1) Configure API Keys
Enter your `OPENROUTER_API_KEY` and `PINECONE_API_KEY` when prompted.
Do not hard-code either key into the notebook.


In [ ]:
import os
from getpass import getpass

# Enter keys securely when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
PINECONE_API_KEY = getpass("Enter your Pinecone API key: ")


In [ ]:
from openai import OpenAI

# OpenRouter provides an OpenAI-compatible API.
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

from pinecone import Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)


### 2) Create a tiny corpus
We'll use three short documents to keep things simple.


In [ ]:
documents = [
    {"id": "doc-1", "text": "Pinecone is a vector database for fast similarity search over embeddings."},
    {"id": "doc-2", "text": "Retrieval-Augmented Generation (RAG) combines search with generation to reduce hallucinations."},
    {"id": "doc-3", "text": "OpenRouter provides access to multiple AI models through a unified API."},
]
len(documents)


### 3) Ask the LLM without retrieval (vanilla)
We'll ask a question directly and see the answer, which might be generic or hallucinated.


In [ ]:
MODEL = "openai/gpt-4o-mini"
question = "What is RAG and how could Pinecone help implement it?"

vanilla = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": question},
    ],
)
print(vanilla.choices[0].message.content)


### 4) Embed documents
We'll generate embeddings using an embedding model available through OpenRouter.


In [ ]:
EMBED_MODEL = "openai/text-embedding-3-small"

def embed_text(text: str):
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return resp.data[0].embedding


In [ ]:
vectors = []
for doc in documents:
    vectors.append({
        "id": doc["id"],
        "values": embed_text(doc["text"]),
        "metadata": {"text": doc["text"]},
    })
len(vectors), len(vectors[0]["values"])


### 5) Create an index in Pinecone and upsert vectors
We'll create a small index with cosine similarity and insert our vectors.


In [ ]:
from pinecone import ServerlessSpec

In [ ]:
INDEX_NAME = "demo-rag-index-sarasai"
NAMESPACE = "ns1"
DIMENSION = len(vectors[0]["values"])  # should match embedding size

# Create index if it doesn't exist
existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )


In [ ]:
index = pc.Index(INDEX_NAME)
# Upsert our vectors
up = index.upsert(vectors=vectors, namespace="ns1")
print("Upsert response:", up)

stats = index.describe_index_stats()
ns_count = stats.get("namespaces", {}).get("ns1", {}).get("vector_count", stats.get("total_vector_count", 0))
print("Index 'ns1' count:", ns_count)

### 6) Retrieve top-k documents for the question
We'll embed the question and query Pinecone for the most similar chunks.


In [ ]:
question

In [ ]:
def retrieve(query: str, top_k: int = 2):
    query_vec = embed_text(query)
    res = index.query(vector=query_vec, top_k=top_k, include_metadata=True, namespace="ns1")
    return [m["metadata"]["text"] for m in res["matches"]]

retrieved = retrieve(question, top_k=2)
retrieved


### 7) Answer with retrieved context (RAG)
We prepend the retrieved snippets to the prompt to ground the model's answer.


In [ ]:
context = "\n\n".join(retrieved)
prompt = f"Answer the question using only the context.\n\nContext:\n{context}\n\nQuestion: {question}"

rag = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer using only the provided context. If unsure, say you don't know."},
        {"role": "user", "content": prompt},
    ],
)
print(rag.choices[0].message.content)
